# 03 · 剩余关键数据补齐 Notebook

本 notebook 用于补齐银河证券 ETF 风格轮动策略复现仍缺的数据：全市场个股日频数据、历史指数成分权重、历史行业分类、财务 PIT 字段、ETF 复权行情/净值、宏观指标断档，以及 parquet 兼容性修复。

设计原则：

- 只生成候选文件，默认不覆盖 `data/raw/*.parquet` 正式文件。
- 所有大规模抓取默认关闭，需要在参数区显式开启。
- 所有抓取按年份、月份、股票/指数批次分片保存，可断点续跑。
- 对公告日/PIT、替代口径、无法获取字段显式标注。
- 不伪造数据；数据源不可用时输出失败原因和下一步建议。

目标候选输出：

- `data/raw/stock_daily_candidate.parquet`
- `data/raw/stock_industry_candidate.parquet`
- `data/raw/index_constituents_candidate.parquet`
- `data/raw/stock_financials_candidate.parquet`
- `data/raw/etf_daily_candidate.parquet`
- `data/raw/macro_<指标名>_candidate.parquet`


## Section 0: 环境和路径

设置项目路径、运行参数、依赖检查，并读取策略、指数池和宏观指标配置。

In [6]:
import os
import sys
import time
import json
import math
import shutil
import warnings
import importlib.util
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RAW = PROJECT_ROOT / "data" / "raw"
CONFIG = PROJECT_ROOT / "config"
NOTEBOOKS = PROJECT_ROOT / "notebooks"
RAW.mkdir(parents=True, exist_ok=True)

START_DATE = "2014-01-01"
END_DATE = "2026-04-17"
MIN_CORE_START = "2020-01-01"

# 安全开关：默认不进行大规模下载、不覆盖正式 raw 文件。
RUN_STOCK_DAILY_FETCH = False
RUN_STOCK_INDUSTRY_FETCH = False
RUN_INDEX_CONSTITUENTS_FETCH = False
RUN_FINANCIALS_FETCH = False
RUN_ETF_FETCH = False
RUN_MACRO_FETCH = False
RUN_PARQUET_REPAIR = False
OVERWRITE_CANONICAL = False

# 抓取参数。
BATCH_SIZE = 50
RETRY = 3
SLEEP_SECONDS = 0.35
FETCH_DYNAMIC_INDEX_POOL_FROM_ETF = True

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW =", RAW)
print("时间范围 =", START_DATE, "至", END_DATE)

required_packages = [
    "pandas", "pyarrow", "duckdb", "akshare", "tushare", "baostock", "requests", "tqdm", "yaml"
]
package_status = []
for pkg in required_packages:
    import_name = "yaml" if pkg == "yaml" else pkg
    package_status.append({"package": pkg, "available": importlib.util.find_spec(import_name) is not None})
pd.DataFrame(package_status)


PROJECT_ROOT = /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3
RAW = /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw
时间范围 = 2014-01-01 至 2026-04-17


,package,available
0,pandas,True
1,pyarrow,True
2,duckdb,True
3,akshare,True
4,tushare,True
5,baostock,True
6,requests,True
7,tqdm,True
8,yaml,True


In [7]:
import yaml

def load_yaml(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

strategy_cfg = load_yaml(CONFIG / "strategy.yaml")
index_cfg = load_yaml(CONFIG / "index_universe.yaml")
macro_cfg = load_yaml(CONFIG / "macro_indicators.yaml")

START_DATE = strategy_cfg.get("backtest", {}).get("data_lookback_start", START_DATE)
END_DATE = strategy_cfg.get("backtest", {}).get("end_date", END_DATE)

print("strategy backtest:", strategy_cfg.get("backtest", {}))
print("index mode:", index_cfg.get("mode"))
print("macro categories:", list(macro_cfg.get("categories", {}).keys()))


strategy backtest: {'start_date': '2020-01-02', 'end_date': '2026-04-17', 'extended_start_date': '2016-01-04', 'data_lookback_start': '2014-01-01', 'initial_capital': 100000000, 'cost_rate': 0.0003, 'benchmark_index': '000985.CSI', 'rebalance': 'W-LAST', 'execution_lag': 1}
index mode: static
macro categories: ['consumption', 'export', 'industrial', 'credit', 'fx']


In [8]:
def import_optional(package_name, import_name=None):
    import_name = import_name or package_name
    try:
        return __import__(import_name), None
    except Exception as exc:
        return None, repr(exc)

ak, ak_err = import_optional("akshare")
ts, ts_err = import_optional("tushare")
bs, bs_err = import_optional("baostock")
duckdb, duckdb_err = import_optional("duckdb")
requests, requests_err = import_optional("requests")

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

OPTIONAL_IMPORTS = {
    "akshare": ak_err,
    "tushare": ts_err,
    "baostock": bs_err,
    "duckdb": duckdb_err,
    "requests": requests_err,
}
OPTIONAL_IMPORTS


{'akshare': None,
 'tushare': None,
 'baostock': None,
 'duckdb': None,
 'requests': None}

## Section 1: 当前数据体检

检查 raw 数据文件是否存在、pandas/DuckDB 是否可读、行数、字段、日期范围、证券数量以及关键字段非空率。

In [9]:
def safe_read_parquet_pandas(path, columns=None):
    try:
        return pd.read_parquet(path, columns=columns), None
    except Exception as exc:
        return None, repr(exc)

def safe_read_parquet_duckdb(path):
    if duckdb is None:
        return None, duckdb_err or "duckdb not installed"
    try:
        con = duckdb.connect(database=":memory:")
        df = con.execute("SELECT * FROM read_parquet(?)", [str(path)]).df()
        con.close()
        return df, None
    except Exception as exc:
        return None, repr(exc)

def nonnull_rate(df, cols):
    out = {}
    for c in cols:
        if c in df.columns:
            out[c] = round(float(df[c].notna().mean()), 4) if len(df) else None
    return out

def detect_date_columns(df):
    candidates = [c for c in df.columns if c.lower() in {"date", "trade_date", "report_date", "ann_date", "publish_date"} or "date" in c.lower()]
    return candidates

def summarize_df(df):
    if df is None:
        return {}
    date_cols = detect_date_columns(df)
    date_min, date_max = None, None
    if date_cols:
        d = pd.to_datetime(df[date_cols[0]], errors="coerce")
        if d.notna().any():
            date_min = d.min().date().isoformat()
            date_max = d.max().date().isoformat()
    counts = {}
    for col in ["code", "index_code", "etf_code"]:
        if col in df.columns:
            counts[f"n_{col}"] = int(df[col].nunique(dropna=True))
    key_cols = ["close_adj", "total_mv", "float_mv", "turnover", "amount", "industry", "weight", "eps_ttm", "net_profit_ttm", "nav", "close", "value", "tracking_index"]
    return {
        "rows": len(df),
        "columns": ", ".join(map(str, df.columns.tolist())),
        "date_min": date_min,
        "date_max": date_max,
        **counts,
        "nonnull_rate": json.dumps(nonnull_rate(df, key_cols), ensure_ascii=False),
    }

def summarize_parquet_file(path):
    path = Path(path)
    row = {"file": path.name, "path": str(path.relative_to(PROJECT_ROOT)), "exists": path.exists()}
    if not path.exists():
        row.update({"pandas_readable": False, "duckdb_readable": False, "error": "missing"})
        return row
    df, err = safe_read_parquet_pandas(path)
    row["pandas_readable"] = err is None
    row["pandas_error"] = err
    if df is None:
        df_duck, derr = safe_read_parquet_duckdb(path)
        row["duckdb_readable"] = derr is None
        row["duckdb_error"] = derr
        df = df_duck
    else:
        row["duckdb_readable"] = None
        row["duckdb_error"] = None
    row.update(summarize_df(df))
    return row

canonical_files = [
    "trade_calendar.parquet", "stock_universe.parquet", "stock_daily.parquet", "stock_industry.parquet",
    "stock_financials.parquet", "index_daily.parquet", "index_constituents.parquet", "etf_info.parquet", "etf_daily.parquet",
]
macro_files = sorted(p.name for p in RAW.glob("macro_*.parquet"))
status_rows = [summarize_parquet_file(RAW / f) for f in canonical_files + macro_files]
data_status = pd.DataFrame(status_rows)
data_status


,file,path,exists,pandas_readable,pandas_error,duckdb_readable,duckdb_error,rows,columns,date_min,date_max,nonnull_rate,n_code,error
0,trade_calendar.parquet,data/raw/trade_calendar.parquet,True,True,NaN,None,NaN,2987.0,date,2014-01-02,2026-04-17,{},NaN,NaN
1,stock_universe.parquet,data/raw/stock_universe.parquet,True,True,NaN,None,NaN,5420.0,"code, name, list_date, delist_date",1990-12-01,2025-12-31,{},5420.0,NaN
2,stock_daily.parquet,data/raw/stock_daily.parquet,True,True,NaN,None,NaN,1881421.0,"date, code, close_adj, total_mv, float_mv, tur...",2024-01-02,2026-04-17,"{""close_adj"": 1.0, ""total_mv"": 1.0, ""float_mv""...",5417.0,NaN
3,stock_industry.parquet,data/raw/stock_industry.parquet,True,True,NaN,None,NaN,802160.0,"date, code, industry",2014-01-30,2026-04-17,"{""industry"": 0.0}",5420.0,NaN
4,stock_financials.parquet,data/raw/stock_financials.parquet,True,True,NaN,None,NaN,308940.0,"report_date, code, dtoa, revenue, net_profit, ...",2012-03-31,2026-03-31,"{""eps_ttm"": 0.0}",5420.0,NaN
5,index_daily.parquet,data/raw/index_daily.parquet,True,True,NaN,None,NaN,48814.0,"date, code, close",2014-01-02,2026-04-17,"{""close"": 1.0}",18.0,NaN
6,index_constituents.parquet,data/raw/index_constituents.parquet,False,False,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing
7,etf_info.parquet,data/raw/etf_info.parquet,True,True,NaN,None,NaN,1566.0,"code, name, track_raw, track_name, tracking_in...",2014-01-02,2026-02-02,"{""tracking_index"": 0.1501}",1566.0,NaN
8,etf_daily.parquet,data/raw/etf_daily.parquet,True,True,NaN,None,NaN,701945.0,"date, code, nav, close, amount",2014-01-02,2026-04-17,"{""amount"": 0.3237, ""nav"": 0.0, ""close"": 0.3238}",235.0,NaN
9,macro_DR007.parquet,data/raw/macro_DR007.parquet,True,True,NaN,None,NaN,2216.0,"date, value",2017-05-31,2026-04-17,"{""value"": 1.0}",NaN,NaN


In [10]:
def write_status_table(df, name="data_status_report.csv"):
    out = RAW / name
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print("saved", out)

write_status_table(data_status)


saved /Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3/data/raw/data_status_report.csv


## Section 2: parquet 修复工具

`repair_parquet_with_duckdb(path, backup=True)`：pandas 可读则跳过；pandas 不可读但 DuckDB 可读则先备份，再用 DuckDB 读取并用 pandas/pyarrow 重写。默认仅在 `RUN_PARQUET_REPAIR=True` 时批量执行。

In [11]:
def repair_parquet_with_duckdb(path, backup=True, overwrite=True):
    path = Path(path)
    log = {"path": str(path), "exists": path.exists(), "action": None, "ok": False, "message": None}
    if not path.exists():
        log.update(action="skip", message="file missing")
        return log

    _, pd_err = safe_read_parquet_pandas(path)
    if pd_err is None:
        log.update(action="skip", ok=True, message="pandas readable")
        return log

    df, duck_err = safe_read_parquet_duckdb(path)
    if duck_err is not None:
        log.update(action="failed", message=f"pandas error={pd_err}; duckdb error={duck_err}")
        return log

    backup_path = None
    if backup:
        backup_dir = RAW / "_backup_corrupt_parquet"
        backup_dir.mkdir(parents=True, exist_ok=True)
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_path = backup_dir / f"{path.stem}.{stamp}{path.suffix}"
        shutil.copy2(path, backup_path)

    target = path if overwrite else path.with_name(path.stem + "_fixed.parquet")
    df.to_parquet(target, index=False, engine="pyarrow")
    _, check_err = safe_read_parquet_pandas(target)
    log.update(
        action="rewrite" if overwrite else "write_fixed",
        ok=check_err is None,
        message="repaired" if check_err is None else f"rewrite failed check: {check_err}",
        backup=str(backup_path) if backup_path else None,
        target=str(target),
        rows=len(df),
    )
    return log

repair_targets = [RAW / f for f in canonical_files + macro_files if (RAW / f).exists()]
repair_logs = []
if RUN_PARQUET_REPAIR:
    for path in tqdm(repair_targets, desc="repair parquet"):
        repair_logs.append(repair_parquet_with_duckdb(path, backup=True, overwrite=True))
else:
    print("RUN_PARQUET_REPAIR=False，未执行修复。可单独调用 repair_parquet_with_duckdb(path)。")

pd.DataFrame(repair_logs)


RUN_PARQUET_REPAIR=False，未执行修复。可单独调用 repair_parquet_with_duckdb(path)。


""


## 通用工具函数

以下函数供 Section 3-9 复用：日期、股票代码格式、分片保存、失败重试、候选文件合并、覆盖率报告。

In [12]:
def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def sanitize_filename(name):
    return str(name).replace("/", "_").replace("\\", "_").replace(":", "_").replace("*", "_").replace("?", "_").replace('"', "_").replace("<", "_").replace(">", "_").replace("|", "_").replace(" ", "")

def to_yyyymmdd(x):
    return pd.to_datetime(x).strftime("%Y%m%d")

def to_date_str(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")

def year_ranges(start=START_DATE, end=END_DATE):
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    years = range(start_ts.year, end_ts.year + 1)
    for y in years:
        s = max(start_ts, pd.Timestamp(f"{y}-01-01"))
        e = min(end_ts, pd.Timestamp(f"{y}-12-31"))
        yield y, s.strftime("%Y-%m-%d"), e.strftime("%Y-%m-%d")

def month_end_dates(start=START_DATE, end=END_DATE, trade_calendar=None):
    start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
    if trade_calendar is not None and len(trade_calendar):
        d = pd.to_datetime(trade_calendar)
        d = d[(d >= start_ts) & (d <= end_ts)]
        return pd.Series(d).groupby([d.year, d.month]).max().dt.strftime("%Y-%m-%d").tolist()
    return pd.date_range(start_ts, end_ts, freq="M").strftime("%Y-%m-%d").tolist()

def chunks(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield i // size, seq[i:i + size]

def retry_call(fn, retry=RETRY, sleep=SLEEP_SECONDS, *args, **kwargs):
    last = None
    for attempt in range(1, retry + 1):
        try:
            return fn(*args, **kwargs), None
        except Exception as exc:
            last = exc
            time.sleep(sleep * attempt)
    return None, repr(last)

def normalize_a_code(code, for_source=None):
    s = str(code).strip().upper().replace(" ", "")
    if "." in s:
        raw, exch = s.split(".", 1)
    elif s.endswith("SH") or s.endswith("SZ") or s.endswith("BJ"):
        raw, exch = s[:6], s[6:]
    else:
        raw = s[:6]
        exch = "SH" if raw.startswith(("6", "9")) else "BJ" if raw.startswith(("8", "4")) else "SZ"
    if for_source == "akshare_symbol":
        return raw
    if for_source == "tushare":
        return f"{raw}.{exch}"
    if for_source == "baostock":
        return f"sh.{raw}" if exch == "SH" else f"sz.{raw}"
    return f"{raw}.{exch}"

def normalize_index_code(code, for_source=None):
    s = str(code).strip().upper().replace(" ", "")
    if "." in s:
        raw, exch = s.split(".", 1)
    elif s.endswith(("SH", "SZ", "CSI", "BJ")):
        raw, exch = s.split(".", 1) if "." in s else (s[:6], s[6:])
    else:
        raw = s[:6]
        exch = "SH" if raw.startswith("000") or raw.startswith("932") else "SZ"
    if for_source == "akshare_em":
        # 东方财富常用 market 前缀：1=沪/中证部分，0=深/北部分。不同接口可能不同，失败表会记录。
        market = "1" if exch in {"SH", "CSI"} else "0"
        return f"{market}.{raw}"
    if for_source == "tushare":
        return f"{raw}.{exch}"
    return f"{raw}.{exch}"

def read_existing_codes():
    universe_path = RAW / "stock_universe.parquet"
    daily_path = RAW / "stock_daily.parquet"
    for path in [universe_path, daily_path]:
        if path.exists():
            df, err = safe_read_parquet_pandas(path)
            if err is None and "code" in df.columns:
                return sorted(df["code"].dropna().astype(str).map(normalize_a_code).unique())
    return []

def read_trade_calendar_dates():
    path = RAW / "trade_calendar.parquet"
    if path.exists():
        df, err = safe_read_parquet_pandas(path)
        if err is None:
            for c in ["date", "trade_date"]:
                if c in df.columns:
                    return pd.to_datetime(df[c], errors="coerce").dropna().sort_values().tolist()
    return []

def merge_partitions(parts_dir, output_path, required_columns=None, dedupe_keys=None):
    parts = sorted(Path(parts_dir).glob("*.parquet"))
    frames = []
    bad = []
    for p in tqdm(parts, desc=f"merge {Path(parts_dir).name}"):
        df, err = safe_read_parquet_pandas(p)
        if err is None:
            frames.append(df)
        else:
            bad.append({"part": str(p), "error": err})
    if not frames:
        print("no readable parts", parts_dir)
        return pd.DataFrame(), pd.DataFrame(bad)
    out = pd.concat(frames, ignore_index=True)
    if required_columns:
        for c in required_columns:
            if c not in out.columns:
                out[c] = pd.NA
        out = out[required_columns + [c for c in out.columns if c not in required_columns]]
    if dedupe_keys:
        out = out.drop_duplicates(dedupe_keys, keep="last")
    out.to_parquet(output_path, index=False, engine="pyarrow")
    print("saved", output_path, "rows=", len(out), "bad_parts=", len(bad))
    return out, pd.DataFrame(bad)

def coverage_report(df, date_col="date", key_cols=None, start=START_DATE, end=END_DATE):
    if df is None or len(df) == 0:
        return {"rows": 0, "start": None, "end": None, "nonnull_rate": {}}
    d = pd.to_datetime(df[date_col], errors="coerce") if date_col in df.columns else pd.Series(dtype="datetime64[ns]")
    key_cols = key_cols or [c for c in df.columns if c not in {date_col}]
    return {
        "rows": len(df),
        "start": d.min().date().isoformat() if d.notna().any() else None,
        "end": d.max().date().isoformat() if d.notna().any() else None,
        "covers_2020_2026": bool(d.min() <= pd.Timestamp(MIN_CORE_START) and d.max() >= pd.Timestamp(end)) if d.notna().any() else False,
        "covers_2014_2026": bool(d.min() <= pd.Timestamp(start) and d.max() >= pd.Timestamp(end)) if d.notna().any() else False,
        "nonnull_rate": nonnull_rate(df, key_cols),
    }


## Section 3: 补齐 stock_daily

目标 schema：`date, code, close_adj, total_mv, float_mv, turnover, amount`。

数据源优先级：Choice 本地环境、Tushare Pro、AkShare 东方财富、BaoStock。默认按年份和股票代码批次保存到 `data/raw/stock_daily_parts_remaining/`，最后合并为 `stock_daily_candidate.parquet`。市值字段公开源不稳定，无法获取时保留空值并输出缺口报告。

In [13]:
STOCK_DAILY_COLUMNS = ["date", "code", "close_adj", "total_mv", "float_mv", "turnover", "amount"]
STOCK_DAILY_PARTS = ensure_dir(RAW / "stock_daily_parts_remaining")

TUSHARE_TOKEN = os.environ.get("TUSHARE_TOKEN", "")
CHOICE_AVAILABLE = False
try:
    import EmQuantAPI  # noqa: F401
    CHOICE_AVAILABLE = True
except Exception:
    CHOICE_AVAILABLE = False

stock_codes = read_existing_codes()
print("stock code count from local files:", len(stock_codes))
print("Choice available:", CHOICE_AVAILABLE, "Tushare token:", bool(TUSHARE_TOKEN), "AkShare:", ak is not None, "BaoStock:", bs is not None)


stock code count from local files: 5420
Choice available: True Tushare token: False AkShare: True BaoStock: True


In [14]:
def fetch_stock_daily_akshare_one(code, start_date, end_date):
    if ak is None:
        raise RuntimeError("akshare not installed")
    symbol = normalize_a_code(code, "akshare_symbol")
    df = ak.stock_zh_a_hist(symbol=symbol, period="daily", start_date=to_yyyymmdd(start_date), end_date=to_yyyymmdd(end_date), adjust="qfq")
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=STOCK_DAILY_COLUMNS)
    rename = {"日期": "date", "收盘": "close_adj", "成交额": "amount", "换手率": "turnover"}
    df = df.rename(columns=rename)
    out = pd.DataFrame()
    out["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    out["code"] = normalize_a_code(code)
    out["close_adj"] = pd.to_numeric(df.get("close_adj"), errors="coerce")
    out["turnover"] = pd.to_numeric(df.get("turnover"), errors="coerce")
    out["amount"] = pd.to_numeric(df.get("amount"), errors="coerce")
    out["total_mv"] = pd.NA
    out["float_mv"] = pd.NA
    return out[STOCK_DAILY_COLUMNS]

def fetch_stock_daily_baostock_one(code, start_date, end_date):
    if bs is None:
        raise RuntimeError("baostock not installed")
    bs_code = normalize_a_code(code, "baostock")
    fields = "date,code,close,turn,amount"
    rs = bs.query_history_k_data_plus(bs_code, fields, start_date=start_date, end_date=end_date, frequency="d", adjustflag="2")
    rows = []
    while rs.error_code == "0" and rs.next():
        rows.append(rs.get_row_data())
    if rs.error_code != "0":
        raise RuntimeError(f"baostock error {rs.error_code}: {rs.error_msg}")
    df = pd.DataFrame(rows, columns=fields.split(","))
    if df.empty:
        return pd.DataFrame(columns=STOCK_DAILY_COLUMNS)
    out = pd.DataFrame({
        "date": pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d"),
        "code": normalize_a_code(code),
        "close_adj": pd.to_numeric(df["close"], errors="coerce"),
        "total_mv": pd.NA,
        "float_mv": pd.NA,
        "turnover": pd.to_numeric(df["turn"], errors="coerce"),
        "amount": pd.to_numeric(df["amount"], errors="coerce"),
    })
    return out[STOCK_DAILY_COLUMNS]

def fetch_stock_daily_tushare_one(code, start_date, end_date):
    if ts is None or not TUSHARE_TOKEN:
        raise RuntimeError("tushare not installed or TUSHARE_TOKEN missing")
    pro = ts.pro_api(TUSHARE_TOKEN)
    ts_code = normalize_a_code(code, "tushare")
    daily = pro.daily(ts_code=ts_code, start_date=to_yyyymmdd(start_date), end_date=to_yyyymmdd(end_date))
    adj = pro.adj_factor(ts_code=ts_code, start_date=to_yyyymmdd(start_date), end_date=to_yyyymmdd(end_date))
    basic = pro.daily_basic(ts_code=ts_code, start_date=to_yyyymmdd(start_date), end_date=to_yyyymmdd(end_date), fields="ts_code,trade_date,turnover_rate,total_mv,circ_mv")
    if daily is None or daily.empty:
        return pd.DataFrame(columns=STOCK_DAILY_COLUMNS)
    df = daily.merge(adj, on=["ts_code", "trade_date"], how="left").merge(basic, on=["ts_code", "trade_date"], how="left")
    df["close_adj"] = pd.to_numeric(df["close"], errors="coerce") * pd.to_numeric(df["adj_factor"], errors="coerce")
    out = pd.DataFrame({
        "date": pd.to_datetime(df["trade_date"], errors="coerce").dt.strftime("%Y-%m-%d"),
        "code": normalize_a_code(code),
        "close_adj": df["close_adj"],
        "total_mv": pd.to_numeric(df.get("total_mv"), errors="coerce") * 10000,
        "float_mv": pd.to_numeric(df.get("circ_mv"), errors="coerce") * 10000,
        "turnover": pd.to_numeric(df.get("turnover_rate"), errors="coerce"),
        "amount": pd.to_numeric(df.get("amount"), errors="coerce") * 1000,
    })
    return out[STOCK_DAILY_COLUMNS]

def fetch_stock_daily_choice_one(code, start_date, end_date):
    raise NotImplementedError("Choice 本地接口字段和登录状态依赖终端环境。请在此函数内接入 css/csd，并输出 STOCK_DAILY_COLUMNS。")

def fetch_stock_daily_one(code, start_date, end_date):
    errors = []
    for name, fn, enabled in [
        ("choice", fetch_stock_daily_choice_one, CHOICE_AVAILABLE),
        ("tushare", fetch_stock_daily_tushare_one, bool(TUSHARE_TOKEN and ts is not None)),
        ("akshare", fetch_stock_daily_akshare_one, ak is not None),
        ("baostock", fetch_stock_daily_baostock_one, bs is not None),
    ]:
        if not enabled:
            continue
        try:
            df = fn(code, start_date, end_date)
            if df is not None and len(df):
                df["source"] = name
                return df, None
            errors.append(f"{name}: empty")
        except Exception as exc:
            errors.append(f"{name}: {repr(exc)}")
    return pd.DataFrame(columns=STOCK_DAILY_COLUMNS + ["source"]), "; ".join(errors) or "no source enabled"


In [15]:
def fetch_stock_daily_parts(codes, start=START_DATE, end=END_DATE, batch_size=BATCH_SIZE):
    failures = []
    if bs is not None:
        try:
            bs.login()
        except Exception as exc:
            failures.append({"stage": "baostock_login", "error": repr(exc)})
    try:
        for year, y_start, y_end in year_ranges(start, end):
            for batch_id, batch_codes in chunks(codes, batch_size):
                part_path = STOCK_DAILY_PARTS / f"part_{batch_id:04d}_{year}.parquet"
                fail_path = STOCK_DAILY_PARTS / f"part_{batch_id:04d}_{year}.failures.csv"
                if part_path.exists():
                    continue
                frames = []
                for code in tqdm(batch_codes, desc=f"stock_daily {year} batch {batch_id}"):
                    df, err = retry_call(fetch_stock_daily_one, code=code, start_date=y_start, end_date=y_end)
                    if df is not None and len(df):
                        frames.append(df)
                    if err:
                        failures.append({"year": year, "batch": batch_id, "code": code, "error": err})
                    time.sleep(SLEEP_SECONDS)
                if frames:
                    part = pd.concat(frames, ignore_index=True)
                    part.to_parquet(part_path, index=False, engine="pyarrow")
                if failures:
                    pd.DataFrame(failures).to_csv(fail_path, index=False, encoding="utf-8-sig")
    finally:
        if bs is not None:
            try:
                bs.logout()
            except Exception:
                pass
    return pd.DataFrame(failures)

if RUN_STOCK_DAILY_FETCH:
    if not stock_codes:
        raise RuntimeError("无法从本地 stock_universe/stock_daily 获取股票代码，请先准备股票池。")
    stock_daily_failures = fetch_stock_daily_parts(stock_codes)
else:
    print("RUN_STOCK_DAILY_FETCH=False，未执行 stock_daily 抓取。")
    stock_daily_failures = pd.DataFrame()

stock_daily_failures.head()


RUN_STOCK_DAILY_FETCH=False，未执行 stock_daily 抓取。


""


In [16]:
stock_daily_candidate_path = RAW / "stock_daily_candidate.parquet"
stock_daily_candidate, stock_daily_bad_parts = merge_partitions(
    STOCK_DAILY_PARTS,
    stock_daily_candidate_path,
    required_columns=STOCK_DAILY_COLUMNS + ["source"],
    dedupe_keys=["date", "code"],
) if any(STOCK_DAILY_PARTS.glob("*.parquet")) else (pd.DataFrame(), pd.DataFrame())

coverage_report(stock_daily_candidate, key_cols=STOCK_DAILY_COLUMNS)


{'rows': 0, 'start': None, 'end': None, 'nonnull_rate': {}}

## Section 4: 补齐 stock_industry

目标 schema：`date, code, industry`。优先历史行业快照；如果只能拿当前行业，字段 `pit_quality` 会标记为 `approx_current_industry`，不得当成严格历史口径使用。

In [ ]:
STOCK_INDUSTRY_COLUMNS = ["date", "code", "industry"]
STOCK_INDUSTRY_PARTS = ensure_dir(RAW / "stock_industry_parts_remaining")
trade_dates = read_trade_calendar_dates()
month_ends = month_end_dates(START_DATE, END_DATE, trade_dates)
print("month end count:", len(month_ends), "first/last:", (month_ends[:1], month_ends[-1:] if month_ends else []))


In [ ]:
def fetch_industry_tushare_snapshot(date, codes=None):
    if ts is None or not TUSHARE_TOKEN:
        raise RuntimeError("tushare not installed or TUSHARE_TOKEN missing")
    pro = ts.pro_api(TUSHARE_TOKEN)
    # Tushare stock_basic 主要是当前行业，不能保证历史 PIT。
    basic = pro.stock_basic(exchange="", list_status="L", fields="ts_code,industry")
    df = basic.rename(columns={"ts_code": "code"})
    df["code"] = df["code"].map(normalize_a_code)
    df["date"] = date
    if codes:
        wanted = set(map(normalize_a_code, codes))
        df = df[df["code"].isin(wanted)]
    df["pit_quality"] = "approx_current_industry"
    df["source"] = "tushare_stock_basic_current"
    return df[["date", "code", "industry", "pit_quality", "source"]]

def fetch_industry_akshare_snapshot(date, codes=None):
    if ak is None:
        raise RuntimeError("akshare not installed")
    # AkShare 行业接口多为当前分类或板块成分，历史快照不可稳定保证。
    df = ak.stock_board_industry_name_em()
    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "code", "industry", "pit_quality", "source"])
    rows = []
    # 这里只生成任务提示，不批量逐行业拉取，避免误触发大量请求。
    raise NotImplementedError("AkShare 行业当前分类需逐行业调用 stock_board_industry_cons_em，默认不作为历史快照。建议优先使用 Choice/Wind/Tushare 历史行业接口。")

def fetch_industry_choice_snapshot(date, codes=None):
    raise NotImplementedError("请接入 Choice 历史中信/申万一级行业接口，输出 date, code, industry。")

def fetch_industry_snapshot(date, codes=None):
    errors = []
    for name, fn, enabled in [
        ("choice", fetch_industry_choice_snapshot, CHOICE_AVAILABLE),
        ("tushare", fetch_industry_tushare_snapshot, bool(TUSHARE_TOKEN and ts is not None)),
        ("akshare", fetch_industry_akshare_snapshot, False),
    ]:
        if not enabled:
            continue
        try:
            df = fn(date, codes=codes)
            if df is not None and len(df):
                return df, None
            errors.append(f"{name}: empty")
        except Exception as exc:
            errors.append(f"{name}: {repr(exc)}")
    return pd.DataFrame(columns=["date", "code", "industry", "pit_quality", "source"]), "; ".join(errors) or "no source enabled"

def fetch_stock_industry_parts(dates, codes=None):
    failures = []
    for d in tqdm(dates, desc="stock_industry month-end"):
        part_path = STOCK_INDUSTRY_PARTS / f"industry_{d}.parquet"
        if part_path.exists():
            continue
        df, err = retry_call(fetch_industry_snapshot, date=d, codes=codes)
        if df is not None and len(df):
            df.to_parquet(part_path, index=False, engine="pyarrow")
        if err:
            failures.append({"date": d, "error": err})
        time.sleep(SLEEP_SECONDS)
    return pd.DataFrame(failures)

if RUN_STOCK_INDUSTRY_FETCH:
    stock_industry_failures = fetch_stock_industry_parts(month_ends, codes=stock_codes or None)
else:
    print("RUN_STOCK_INDUSTRY_FETCH=False，未执行 stock_industry 抓取。")
    stock_industry_failures = pd.DataFrame()

stock_industry_failures.head()


In [ ]:
stock_industry_candidate_path = RAW / "stock_industry_candidate.parquet"
stock_industry_candidate, stock_industry_bad_parts = merge_partitions(
    STOCK_INDUSTRY_PARTS,
    stock_industry_candidate_path,
    required_columns=["date", "code", "industry", "pit_quality", "source"],
    dedupe_keys=["date", "code"],
) if any(STOCK_INDUSTRY_PARTS.glob("*.parquet")) else (pd.DataFrame(), pd.DataFrame())

coverage_report(stock_industry_candidate, key_cols=["industry"])


## Section 5: 补齐 index_constituents

目标 schema：`date, index_code, code, weight`。指数池来自 `config/index_universe.yaml` 的静态池，并可选从 `etf_info.tracking_index` 扩展。优先使用 Choice/Wind/Tushare 指数权重接口；公开源覆盖不足时记录失败表。

In [ ]:
INDEX_CONSTITUENTS_COLUMNS = ["date", "index_code", "code", "weight"]
INDEX_CONSTITUENTS_PARTS = ensure_dir(RAW / "index_constituents_parts_remaining")

def get_static_index_pool():
    return [x["code"] for x in index_cfg.get("static_pool", []) if "code" in x]

def get_dynamic_index_pool_from_etf():
    path = RAW / "etf_info.parquet"
    if not path.exists():
        return []
    df, err = safe_read_parquet_pandas(path)
    if err is not None or "tracking_index" not in df.columns:
        return []
    vals = df["tracking_index"].dropna().astype(str).str.strip()
    vals = vals[vals.ne("")]
    return sorted(vals.unique())

index_pool = sorted(set(get_static_index_pool()) | (set(get_dynamic_index_pool_from_etf()) if FETCH_DYNAMIC_INDEX_POOL_FROM_ETF else set()))
print("index pool count:", len(index_pool))
index_pool[:30]


In [ ]:
def fetch_index_weight_tushare(index_code, date):
    if ts is None or not TUSHARE_TOKEN:
        raise RuntimeError("tushare not installed or TUSHARE_TOKEN missing")
    pro = ts.pro_api(TUSHARE_TOKEN)
    idx = normalize_index_code(index_code, "tushare")
    df = pro.index_weight(index_code=idx, start_date=to_yyyymmdd(date), end_date=to_yyyymmdd(date))
    if df is None or df.empty:
        # 有些接口按月份返回，扩大到当月。
        ts_date = pd.Timestamp(date)
        df = pro.index_weight(index_code=idx, start_date=ts_date.strftime("%Y%m01"), end_date=ts_date.strftime("%Y%m%d"))
    if df is None or df.empty:
        return pd.DataFrame(columns=INDEX_CONSTITUENTS_COLUMNS)
    date_col = "trade_date" if "trade_date" in df.columns else "date"
    weight_col = "weight" if "weight" in df.columns else "con_weight" if "con_weight" in df.columns else None
    code_col = "con_code" if "con_code" in df.columns else "ts_code" if "ts_code" in df.columns else None
    if weight_col is None or code_col is None:
        raise RuntimeError(f"unexpected tushare columns: {df.columns.tolist()}")
    out = pd.DataFrame({
        "date": date,
        "index_code": normalize_index_code(index_code),
        "code": df[code_col].map(normalize_a_code),
        "weight": pd.to_numeric(df[weight_col], errors="coerce"),
        "source": "tushare_index_weight",
    })
    return out[INDEX_CONSTITUENTS_COLUMNS + ["source"]]

def fetch_index_weight_choice(index_code, date):
    raise NotImplementedError("请接入 Choice/Wind 历史指数成分权重接口，输出 date,index_code,code,weight。")

def fetch_index_weight_snapshot(index_code, date):
    errors = []
    for name, fn, enabled in [
        ("choice", fetch_index_weight_choice, CHOICE_AVAILABLE),
        ("tushare", fetch_index_weight_tushare, bool(TUSHARE_TOKEN and ts is not None)),
    ]:
        if not enabled:
            continue
        try:
            df = fn(index_code, date)
            if df is not None and len(df):
                return df, None
            errors.append(f"{name}: empty")
        except Exception as exc:
            errors.append(f"{name}: {repr(exc)}")
    return pd.DataFrame(columns=INDEX_CONSTITUENTS_COLUMNS + ["source"]), "; ".join(errors) or "no source enabled"

def fetch_index_constituents_parts(indices, dates):
    failures = []
    for idx in tqdm(indices, desc="indices"):
        for d in dates:
            part_path = INDEX_CONSTITUENTS_PARTS / f"{sanitize_filename(idx)}_{d}.parquet"
            if part_path.exists():
                continue
            df, err = retry_call(fetch_index_weight_snapshot, index_code=idx, date=d)
            if df is not None and len(df):
                df.to_parquet(part_path, index=False, engine="pyarrow")
            if err:
                failures.append({"date": d, "index_code": idx, "error": err})
            time.sleep(SLEEP_SECONDS)
    return pd.DataFrame(failures)

if RUN_INDEX_CONSTITUENTS_FETCH:
    index_constituents_failures = fetch_index_constituents_parts(index_pool, month_ends)
else:
    print("RUN_INDEX_CONSTITUENTS_FETCH=False，未执行 index_constituents 抓取。")
    index_constituents_failures = pd.DataFrame()

index_constituents_failures.head()


In [ ]:
index_constituents_candidate_path = RAW / "index_constituents_candidate.parquet"
index_constituents_candidate, index_constituents_bad_parts = merge_partitions(
    INDEX_CONSTITUENTS_PARTS,
    index_constituents_candidate_path,
    required_columns=INDEX_CONSTITUENTS_COLUMNS + ["source"],
    dedupe_keys=["date", "index_code", "code"],
) if any(INDEX_CONSTITUENTS_PARTS.glob("*.parquet")) else (pd.DataFrame(), pd.DataFrame())

coverage_report(index_constituents_candidate, key_cols=["weight"])


## Section 6: 补齐财务 PIT 数据

目标字段：`report_date, ann_date, code, dtoa, revenue, net_profit, net_profit_ttm, roa, gpm, ato, bps, eps_ttm, total_assets, total_equity, dividend_ttm, div_yield`。

严格 PIT 依赖公告日 `ann_date`。若数据源无法提供公告日，本节生成 `ann_date_approx` 并标记 `pit_quality='approx_lag'`：一季报/中报/三季报滞后 45 天，年报滞后 120 天。

In [ ]:
FINANCIAL_COLUMNS = [
    "report_date", "ann_date", "ann_date_approx", "code", "dtoa", "revenue", "net_profit", "net_profit_ttm",
    "roa", "gpm", "ato", "bps", "eps_ttm", "total_assets", "total_equity", "dividend_ttm", "div_yield",
]
FINANCIAL_PARTS = ensure_dir(RAW / "stock_financials_parts_remaining")

def approximate_ann_date(report_date):
    d = pd.Timestamp(report_date)
    if d.month == 12 and d.day == 31:
        return (d + pd.Timedelta(days=120)).strftime("%Y-%m-%d")
    return (d + pd.Timedelta(days=45)).strftime("%Y-%m-%d")

def add_ttm_by_code(df, value_col, out_col):
    if value_col not in df.columns:
        df[out_col] = pd.NA
        return df
    df = df.sort_values(["code", "report_date"])
    df[out_col] = df.groupby("code")[value_col].transform(lambda s: pd.to_numeric(s, errors="coerce").rolling(4, min_periods=4).sum())
    return df


In [ ]:
def fetch_financials_tushare_one(code, start=START_DATE, end=END_DATE):
    if ts is None or not TUSHARE_TOKEN:
        raise RuntimeError("tushare not installed or TUSHARE_TOKEN missing")
    pro = ts.pro_api(TUSHARE_TOKEN)
    ts_code = normalize_a_code(code, "tushare")
    start_y, end_y = pd.Timestamp(start).year, pd.Timestamp(end).year
    frames = []
    for y in range(start_y, end_y + 1):
        income = pro.income(ts_code=ts_code, period=f"{y}1231", fields="ts_code,ann_date,f_ann_date,end_date,revenue,n_income_attr_p")
        balancesheet = pro.balancesheet(ts_code=ts_code, period=f"{y}1231", fields="ts_code,ann_date,end_date,total_assets,total_hldr_eqy_exc_min_int")
        fina = pro.fina_indicator(ts_code=ts_code, period=f"{y}1231", fields="ts_code,ann_date,end_date,eps,dt_eps,bps,roe,roa,grossprofit_margin,assets_turn")
        # 年报接口只是兜底。生产环境建议改为按 report_type 或 ann_date 拉全部季报。
        dfs = [x for x in [income, balancesheet, fina] if x is not None and len(x)]
        if not dfs:
            continue
        base = dfs[0]
        for extra in dfs[1:]:
            base = base.merge(extra, on=[c for c in ["ts_code", "ann_date", "end_date"] if c in base.columns and c in extra.columns], how="outer")
        frames.append(base)
        time.sleep(SLEEP_SECONDS)
    if not frames:
        return pd.DataFrame(columns=FINANCIAL_COLUMNS + ["pit_quality", "source"])
    df = pd.concat(frames, ignore_index=True)
    out = pd.DataFrame({
        "report_date": pd.to_datetime(df.get("end_date"), errors="coerce").dt.strftime("%Y-%m-%d"),
        "ann_date": pd.to_datetime(df.get("ann_date"), errors="coerce").dt.strftime("%Y-%m-%d"),
        "code": normalize_a_code(code),
        "revenue": pd.to_numeric(df.get("revenue"), errors="coerce"),
        "net_profit": pd.to_numeric(df.get("n_income_attr_p"), errors="coerce"),
        "roa": pd.to_numeric(df.get("roa"), errors="coerce"),
        "gpm": pd.to_numeric(df.get("grossprofit_margin"), errors="coerce"),
        "ato": pd.to_numeric(df.get("assets_turn"), errors="coerce"),
        "bps": pd.to_numeric(df.get("bps"), errors="coerce"),
        "eps_ttm": pd.to_numeric(df.get("dt_eps"), errors="coerce"),
        "total_assets": pd.to_numeric(df.get("total_assets"), errors="coerce"),
        "total_equity": pd.to_numeric(df.get("total_hldr_eqy_exc_min_int"), errors="coerce"),
    })
    out["ann_date_approx"] = out["report_date"].map(approximate_ann_date)
    out["pit_quality"] = out["ann_date"].notna().map(lambda x: "strict_ann_date" if x else "approx_lag")
    out["dtoa"] = out["total_assets"] / out["total_equity"]
    out["dividend_ttm"] = pd.NA
    out["div_yield"] = pd.NA
    out = add_ttm_by_code(out, "net_profit", "net_profit_ttm")
    out["source"] = "tushare_financials"
    for c in FINANCIAL_COLUMNS:
        if c not in out.columns:
            out[c] = pd.NA
    return out[FINANCIAL_COLUMNS + ["pit_quality", "source"]]

def fetch_financials_choice_one(code, start=START_DATE, end=END_DATE):
    raise NotImplementedError("请接入 Choice 财务与公告日接口。必须输出 ann_date；否则使用 ann_date_approx 并标记近似 PIT。")

def fetch_financials_one(code, start=START_DATE, end=END_DATE):
    errors = []
    for name, fn, enabled in [
        ("choice", fetch_financials_choice_one, CHOICE_AVAILABLE),
        ("tushare", fetch_financials_tushare_one, bool(TUSHARE_TOKEN and ts is not None)),
    ]:
        if not enabled:
            continue
        try:
            df = fn(code, start, end)
            if df is not None and len(df):
                return df, None
            errors.append(f"{name}: empty")
        except Exception as exc:
            errors.append(f"{name}: {repr(exc)}")
    return pd.DataFrame(columns=FINANCIAL_COLUMNS + ["pit_quality", "source"]), "; ".join(errors) or "no source enabled"

def fetch_financials_parts(codes, batch_size=BATCH_SIZE):
    failures = []
    for batch_id, batch_codes in chunks(codes, batch_size):
        part_path = FINANCIAL_PARTS / f"financials_part_{batch_id:04d}.parquet"
        if part_path.exists():
            continue
        frames = []
        for code in tqdm(batch_codes, desc=f"financials batch {batch_id}"):
            df, err = retry_call(fetch_financials_one, code=code)
            if df is not None and len(df):
                frames.append(df)
            if err:
                failures.append({"batch": batch_id, "code": code, "error": err})
            time.sleep(SLEEP_SECONDS)
        if frames:
            pd.concat(frames, ignore_index=True).to_parquet(part_path, index=False, engine="pyarrow")
    return pd.DataFrame(failures)

if RUN_FINANCIALS_FETCH:
    financials_failures = fetch_financials_parts(stock_codes)
else:
    print("RUN_FINANCIALS_FETCH=False，未执行 financials 抓取。")
    financials_failures = pd.DataFrame()

financials_failures.head()


In [ ]:
financials_candidate_path = RAW / "stock_financials_candidate.parquet"
financials_candidate, financials_bad_parts = merge_partitions(
    FINANCIAL_PARTS,
    financials_candidate_path,
    required_columns=FINANCIAL_COLUMNS + ["pit_quality", "source"],
    dedupe_keys=["report_date", "code"],
) if any(FINANCIAL_PARTS.glob("*.parquet")) else (pd.DataFrame(), pd.DataFrame())

coverage_report(financials_candidate, date_col="report_date", key_cols=FINANCIAL_COLUMNS)


## Section 7: 补齐 ETF 数据

目标 schema：`date, code, nav, close, close_adj, amount`。上市前缺失允许存在；上市后应检查 close/amount 完整性。若无法获取 `nav`，优先使用 `close_adj`，否则明确用 `close` 近似执行价。

In [ ]:
ETF_COLUMNS = ["date", "code", "nav", "close", "close_adj", "amount"]
ETF_PARTS = ensure_dir(RAW / "etf_daily_parts_remaining")

def read_etf_info():
    path = RAW / "etf_info.parquet"
    if not path.exists():
        return pd.DataFrame()
    df, err = safe_read_parquet_pandas(path)
    if err is not None:
        print("etf_info read error", err)
        return pd.DataFrame()
    return df

etf_info = read_etf_info()
etf_code_col = "code" if "code" in etf_info.columns else "fund_code" if "fund_code" in etf_info.columns else None
etf_codes = sorted(etf_info[etf_code_col].dropna().astype(str).unique()) if etf_code_col else []
print("ETF count:", len(etf_codes), "columns:", etf_info.columns.tolist() if len(etf_info) else [])


In [ ]:
def fetch_etf_daily_akshare_one(code, start_date, end_date):
    if ak is None:
        raise RuntimeError("akshare not installed")
    symbol = str(code).split(".")[0]
    frames = []
    errors = []
    # 场内基金历史行情，复权收盘价可用 qfq 近似执行价。
    try:
        qfq = ak.fund_etf_hist_em(symbol=symbol, period="daily", start_date=to_yyyymmdd(start_date), end_date=to_yyyymmdd(end_date), adjust="qfq")
        if qfq is not None and len(qfq):
            qfq = qfq.rename(columns={"日期": "date", "收盘": "close_adj", "成交额": "amount"})
            frames.append(qfq[["date", "close_adj", "amount"]])
    except Exception as exc:
        errors.append(f"akshare qfq: {repr(exc)}")
    try:
        raw = ak.fund_etf_hist_em(symbol=symbol, period="daily", start_date=to_yyyymmdd(start_date), end_date=to_yyyymmdd(end_date), adjust="")
        if raw is not None and len(raw):
            raw = raw.rename(columns={"日期": "date", "收盘": "close", "成交额": "amount"})
            frames.append(raw[["date", "close", "amount"]])
    except Exception as exc:
        errors.append(f"akshare raw: {repr(exc)}")
    if not frames:
        raise RuntimeError("; ".join(errors) or "empty")
    out = frames[0]
    for extra in frames[1:]:
        out = out.merge(extra, on="date", how="outer", suffixes=("", "_raw"))
    out["code"] = code
    if "amount_raw" in out.columns:
        out["amount"] = out["amount"].fillna(out["amount_raw"])
    out["nav"] = pd.NA
    out["source"] = "akshare_fund_etf_hist_em"
    for c in ETF_COLUMNS:
        if c not in out.columns:
            out[c] = pd.NA
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    for c in ["nav", "close", "close_adj", "amount"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out[ETF_COLUMNS + ["source"]]

def fetch_etf_daily_choice_one(code, start_date, end_date):
    raise NotImplementedError("请接入 Choice ETF 复权单位净值/复权收盘价接口，输出 ETF_COLUMNS。")

def fetch_etf_daily_one(code, start_date, end_date):
    errors = []
    for name, fn, enabled in [
        ("choice", fetch_etf_daily_choice_one, CHOICE_AVAILABLE),
        ("akshare", fetch_etf_daily_akshare_one, ak is not None),
    ]:
        if not enabled:
            continue
        try:
            df = fn(code, start_date, end_date)
            if df is not None and len(df):
                return df, None
            errors.append(f"{name}: empty")
        except Exception as exc:
            errors.append(f"{name}: {repr(exc)}")
    return pd.DataFrame(columns=ETF_COLUMNS + ["source"]), "; ".join(errors) or "no source enabled"

def fetch_etf_parts(codes, batch_size=200):
    failures = []
    for year, y_start, y_end in year_ranges(START_DATE, END_DATE):
        for batch_id, batch_codes in chunks(codes, batch_size):
            part_path = ETF_PARTS / f"part_{batch_id:04d}_{year}.parquet"
            if part_path.exists():
                continue
            frames = []
            for code in tqdm(batch_codes, desc=f"etf {year} batch {batch_id}"):
                df, err = retry_call(fetch_etf_daily_one, code=code, start_date=y_start, end_date=y_end)
                if df is not None and len(df):
                    frames.append(df)
                if err:
                    failures.append({"year": year, "batch": batch_id, "code": code, "error": err})
                time.sleep(SLEEP_SECONDS)
            if frames:
                pd.concat(frames, ignore_index=True).to_parquet(part_path, index=False, engine="pyarrow")
    return pd.DataFrame(failures)

if RUN_ETF_FETCH:
    etf_failures = fetch_etf_parts(etf_codes)
else:
    print("RUN_ETF_FETCH=False，未执行 ETF 抓取。")
    etf_failures = pd.DataFrame()

etf_failures.head()


In [ ]:
etf_candidate_path = RAW / "etf_daily_candidate.parquet"
etf_candidate, etf_bad_parts = merge_partitions(
    ETF_PARTS,
    etf_candidate_path,
    required_columns=ETF_COLUMNS + ["source"],
    dedupe_keys=["date", "code"],
) if any(ETF_PARTS.glob("*.parquet")) else (pd.DataFrame(), pd.DataFrame())

coverage_report(etf_candidate, key_cols=ETF_COLUMNS)


In [ ]:
def etf_listing_completeness(etf_daily_df, etf_info_df):
    if etf_daily_df is None or etf_daily_df.empty:
        return pd.DataFrame()
    df = etf_daily_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    rows = []
    for code, g in df.groupby("code"):
        first = g["date"].min()
        listed = g[g["date"] >= first]
        rows.append({
            "code": code,
            "first_daily_date": first.date().isoformat() if pd.notna(first) else None,
            "rows": len(g),
            "close_nonnull": round(listed["close"].notna().mean(), 4) if "close" in listed else None,
            "close_adj_nonnull": round(listed["close_adj"].notna().mean(), 4) if "close_adj" in listed else None,
            "nav_nonnull": round(listed["nav"].notna().mean(), 4) if "nav" in listed else None,
            "amount_nonnull": round(listed["amount"].notna().mean(), 4) if "amount" in listed else None,
        })
    return pd.DataFrame(rows)

etf_completeness = etf_listing_completeness(etf_candidate, etf_info)
etf_completeness.head()


## Section 8: 补齐宏观数据

读取 `config/macro_indicators.yaml` 中 enabled 指标，逐个尝试 AkShare、Tushare、官网 requests、手动 CSV。每个指标单独保存候选文件，并输出覆盖报告。轻纺城成交量等公开接口不足的数据可放入 `data/manual/macro_<指标名>.csv`，字段至少包含 `date,value`。

In [ ]:
MACRO_PARTS = ensure_dir(RAW / "macro_parts_remaining")
MANUAL = ensure_dir(PROJECT_ROOT / "data" / "manual")

def flatten_macro_config(cfg):
    rows = []
    for cat_key, cat in cfg.get("categories", {}).items():
        for item in cat.get("indicators", []):
            rows.append({
                "category": cat_key,
                "category_cn": cat.get("cn_name", cat_key),
                "name": item.get("name"),
                "enabled": item.get("enabled", True),
                "freq": item.get("freq"),
                "direction": item.get("direction"),
                "choice_edb_code": item.get("choice_edb_code"),
            })
    return pd.DataFrame(rows)

macro_items = flatten_macro_config(macro_cfg)
macro_items


In [ ]:
def read_manual_macro_csv(name):
    path = MANUAL / f"macro_{sanitize_filename(name)}.csv"
    if not path.exists():
        raise FileNotFoundError(f"manual csv not found: {path}")
    df = pd.read_csv(path)
    if "date" not in df.columns or "value" not in df.columns:
        raise RuntimeError("manual csv must contain date,value")
    out = df[["date", "value"]].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out["source"] = "manual_csv"
    return out.dropna(subset=["date"])

def fetch_macro_akshare(name):
    if ak is None:
        raise RuntimeError("akshare not installed")
    # 已知可用或常见接口映射。部分接口会因 AkShare 版本变化而变动，失败会进入报告。
    if name in {"SHIBOR_1周", "SHIBOR_1年"}:
        df = ak.macro_china_shibor_all()
        date_col = "日期" if "日期" in df.columns else "date"
        col_map = {"SHIBOR_1周": ["1周", "1w", "Shibor:1周"], "SHIBOR_1年": ["1年", "1y", "Shibor:1年"]}
        value_col = next((c for c in col_map[name] if c in df.columns), None)
        if value_col is None:
            raise RuntimeError(f"cannot find SHIBOR column in {df.columns.tolist()}")
        out = df[[date_col, value_col]].rename(columns={date_col: "date", value_col: "value"})
    elif name in {"国债到期收益率_1年", "国债到期收益率_10年"}:
        df = ak.bond_china_yield(start_date=to_yyyymmdd(START_DATE), end_date=to_yyyymmdd(END_DATE))
        date_col = "日期" if "日期" in df.columns else "date"
        candidates = {"国债到期收益率_1年": ["中国国债收益率1年", "1年", "国债1年"], "国债到期收益率_10年": ["中国国债收益率10年", "10年", "国债10年"]}
        value_col = next((c for c in candidates[name] if c in df.columns), None)
        if value_col is None:
            # 宽松匹配。
            value_col = next((c for c in df.columns if ("1年" if name.endswith("1年") else "10年") in str(c) and "国债" in str(c)), None)
        if value_col is None:
            raise RuntimeError(f"cannot find bond yield column in {df.columns.tolist()}")
        out = df[[date_col, value_col]].rename(columns={date_col: "date", value_col: "value"})
    elif name == "中债新综合指数":
        df = ak.bond_new_composite_index_cbond()
        date_col = "日期" if "日期" in df.columns else "date"
        value_col = next((c for c in ["财富指数", "净价指数", "全价指数", "value"] if c in df.columns), None)
        if value_col is None:
            numeric_cols = [c for c in df.columns if c != date_col]
            value_col = numeric_cols[0]
        out = df[[date_col, value_col]].rename(columns={date_col: "date", value_col: "value"})
    elif name == "美元兑人民币_中间价":
        df = ak.currency_boc_sina(symbol="美元", start_date=START_DATE, end_date=END_DATE)
        date_col = "日期" if "日期" in df.columns else "date"
        value_col = next((c for c in ["中行折算价", "现汇卖出价", "value"] if c in df.columns), None)
        if value_col is None:
            raise RuntimeError(f"cannot find FX column in {df.columns.tolist()}")
        out = df[[date_col, value_col]].rename(columns={date_col: "date", value_col: "value"})
    elif name in {"中国出口集装箱运价指数CCFI_综合指数", "中国沿海散货运价指数CBFI_综合指数", "乘用车日均销量_厂家零售_当周", "乘用车日均销量_厂家批发_当周", "电影票房收入", "三峡水库站_出库流量", "三峡水库站_入库流量", "生产资料价格指数", "DR007"}:
        raise NotImplementedError("该指标 AkShare 接口未在本 notebook 中确认稳定，请使用 Choice/Tushare/官网 requests 或 manual_csv。")
    else:
        raise NotImplementedError(f"no akshare mapping for {name}")
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out["source"] = "akshare"
    return out[["date", "value", "source"]].dropna(subset=["date"])

def fetch_macro_tushare(name):
    if ts is None or not TUSHARE_TOKEN:
        raise RuntimeError("tushare not installed or TUSHARE_TOKEN missing")
    raise NotImplementedError("按指标接入 Tushare macro 接口后输出 date,value。")

def fetch_macro_requests(name):
    if requests is None:
        raise RuntimeError("requests not installed")
    raise NotImplementedError("按指标接入官网公开 CSV/JSON 后输出 date,value。")

def fetch_macro_indicator(name):
    errors = []
    for source, fn in [("manual_csv", read_manual_macro_csv), ("akshare", fetch_macro_akshare), ("tushare", fetch_macro_tushare), ("requests", fetch_macro_requests)]:
        try:
            df = fn(name)
            if df is not None and len(df):
                return df, None
            errors.append(f"{source}: empty")
        except Exception as exc:
            errors.append(f"{source}: {repr(exc)}")
    return pd.DataFrame(columns=["date", "value", "source"]), "; ".join(errors)

def save_macro_candidate(name, df):
    out = RAW / f"macro_{sanitize_filename(name)}_candidate.parquet"
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date"]).drop_duplicates("date", keep="last").sort_values("date")
    df.to_parquet(out, index=False, engine="pyarrow")
    return out


In [ ]:
macro_reports = []
macro_failures = []
if RUN_MACRO_FETCH:
    for item in macro_items.to_dict("records"):
        if not item.get("enabled", True):
            macro_reports.append({**item, "status": "disabled", "candidate_file": None})
            continue
        name = item["name"]
        df, err = retry_call(fetch_macro_indicator, name=name)
        if df is not None and len(df):
            out = save_macro_candidate(name, df)
            rep = coverage_report(df, key_cols=["value"])
            macro_reports.append({**item, "status": "ok", "candidate_file": str(out.relative_to(PROJECT_ROOT)), "source": ",".join(sorted(df["source"].dropna().unique())), **rep})
        else:
            macro_failures.append({**item, "error": err})
            macro_reports.append({**item, "status": "failed", "candidate_file": None, "error": err})
        time.sleep(SLEEP_SECONDS)
else:
    print("RUN_MACRO_FETCH=False，未执行宏观抓取。将基于已有正式/候选宏观文件生成覆盖报告。")
    for item in macro_items.to_dict("records"):
        name = item["name"]
        candidate = RAW / f"macro_{sanitize_filename(name)}_candidate.parquet"
        canonical = RAW / f"macro_{name}.parquet"
        path = candidate if candidate.exists() else canonical
        if path.exists():
            df, err = safe_read_parquet_pandas(path)
            if err is None:
                rep = coverage_report(df, key_cols=["value"])
                macro_reports.append({**item, "status": "existing", "candidate_file": str(path.relative_to(PROJECT_ROOT)), **rep})
            else:
                macro_reports.append({**item, "status": "unreadable", "candidate_file": str(path.relative_to(PROJECT_ROOT)), "error": err})
        else:
            macro_reports.append({**item, "status": "missing", "candidate_file": None})

macro_coverage_report = pd.DataFrame(macro_reports)
macro_coverage_report.to_csv(RAW / "macro_coverage_report.csv", index=False, encoding="utf-8-sig")
macro_coverage_report


## Section 9: 最终数据缺口报告

生成总表：`数据项, 目标字段, 当前文件, 候选文件, 是否满足, 日期覆盖, 非空率, 仍缺字段, 是否严格PIT, 备注`。

In [ ]:
TARGETS = [
    {"item": "stock_daily", "target_fields": STOCK_DAILY_COLUMNS, "canonical": RAW / "stock_daily.parquet", "candidate": RAW / "stock_daily_candidate.parquet", "date_col": "date", "strict_pit": "不适用", "note": "需覆盖 2014-2026 全 A；市值字段是 Barra/权重关键字段。"},
    {"item": "index_constituents", "target_fields": INDEX_CONSTITUENTS_COLUMNS, "canonical": RAW / "index_constituents.parquet", "candidate": RAW / "index_constituents_candidate.parquet", "date_col": "date", "strict_pit": "不适用", "note": "硬阻塞：指数风格暴露必须有历史成分股和权重。"},
    {"item": "stock_industry", "target_fields": STOCK_INDUSTRY_COLUMNS, "canonical": RAW / "stock_industry.parquet", "candidate": RAW / "stock_industry_candidate.parquet", "date_col": "date", "strict_pit": "需历史快照", "note": "若 pit_quality=approx_current_industry，只能用于近似复现。"},
    {"item": "stock_financials", "target_fields": FINANCIAL_COLUMNS, "canonical": RAW / "stock_financials.parquet", "candidate": RAW / "stock_financials_candidate.parquet", "date_col": "report_date", "strict_pit": "需 ann_date", "note": "无 ann_date 时使用 ann_date_approx，必须标记近似 PIT。"},
    {"item": "etf_daily", "target_fields": ETF_COLUMNS, "canonical": RAW / "etf_daily.parquet", "candidate": RAW / "etf_daily_candidate.parquet", "date_col": "date", "strict_pit": "不适用", "note": "nav 缺失时使用 close_adj 或 close 近似执行价。"},
]

def evaluate_target(t):
    path = t["candidate"] if t["candidate"].exists() else t["canonical"]
    row = {
        "数据项": t["item"],
        "目标字段": ", ".join(t["target_fields"]),
        "当前文件": str(t["canonical"].relative_to(PROJECT_ROOT)),
        "候选文件": str(t["candidate"].relative_to(PROJECT_ROOT)) if t["candidate"].exists() else "",
        "是否严格PIT": t["strict_pit"],
        "备注": t["note"],
    }
    if not path.exists():
        row.update({"是否满足": False, "日期覆盖": "missing", "非空率": "{}", "仍缺字段": ", ".join(t["target_fields"])})
        return row
    df, err = safe_read_parquet_pandas(path)
    if err is not None:
        row.update({"是否满足": False, "日期覆盖": "unreadable", "非空率": "{}", "仍缺字段": f"unreadable: {err}"})
        return row
    missing = [c for c in t["target_fields"] if c not in df.columns]
    rep = coverage_report(df, date_col=t["date_col"], key_cols=t["target_fields"])
    critical_nonnull = rep.get("nonnull_rate", {})
    if t["item"] == "index_constituents":
        key_ok = critical_nonnull.get("weight", 0) and critical_nonnull.get("weight", 0) > 0.8
    elif t["item"] == "stock_industry":
        key_ok = critical_nonnull.get("industry", 0) and critical_nonnull.get("industry", 0) > 0.8
    elif t["item"] == "stock_financials":
        key_ok = (critical_nonnull.get("ann_date", 0) or critical_nonnull.get("ann_date_approx", 0)) and (critical_nonnull.get("net_profit_ttm", 0) or critical_nonnull.get("eps_ttm", 0))
    elif t["item"] == "etf_daily":
        key_ok = (critical_nonnull.get("close_adj", 0) or critical_nonnull.get("close", 0)) and critical_nonnull.get("amount", 0)
    else:
        key_ok = all(critical_nonnull.get(c, 0) for c in ["close_adj", "amount"] if c in t["target_fields"])
    satisfies = (not missing) and bool(rep.get("covers_2020_2026")) and bool(key_ok)
    row.update({
        "是否满足": satisfies,
        "日期覆盖": f"{rep.get('start')} ~ {rep.get('end')}; 2020-2026={rep.get('covers_2020_2026')}; 2014-2026={rep.get('covers_2014_2026')}",
        "非空率": json.dumps(rep.get("nonnull_rate", {}), ensure_ascii=False),
        "仍缺字段": ", ".join(missing),
    })
    return row

final_gap_rows = [evaluate_target(t) for t in TARGETS]

# 宏观汇总行。
if "macro_coverage_report" in globals() and len(macro_coverage_report):
    enabled_macro = macro_coverage_report[macro_coverage_report.get("enabled", True) == True] if "enabled" in macro_coverage_report.columns else macro_coverage_report
    ok2020 = int(enabled_macro.get("covers_2020_2026", pd.Series(dtype=bool)).fillna(False).sum()) if len(enabled_macro) else 0
    ok2014 = int(enabled_macro.get("covers_2014_2026", pd.Series(dtype=bool)).fillna(False).sum()) if len(enabled_macro) else 0
    final_gap_rows.append({
        "数据项": "macro_*",
        "目标字段": "date, value",
        "当前文件": "data/raw/macro_*.parquet",
        "候选文件": "data/raw/macro_<指标名>_candidate.parquet",
        "是否满足": bool(len(enabled_macro) and ok2020 == len(enabled_macro)),
        "日期覆盖": f"enabled={len(enabled_macro)}, satisfy_2020_2026={ok2020}, satisfy_2014_2026={ok2014}",
        "非空率": "见 macro_coverage_report.csv",
        "仍缺字段": "逐指标检查",
        "是否严格PIT": "不适用",
        "备注": "轻纺城成交量通常需 manual_csv 或官网导出；公开源起点不足需在报告中保留。",
    })

final_gap_report = pd.DataFrame(final_gap_rows)
final_gap_report.to_csv(RAW / "remaining_data_gap_report.csv", index=False, encoding="utf-8-sig")
final_gap_report


In [ ]:
def print_final_conclusion(report):
    unmet = report[~report["是否满足"].astype(bool)] if len(report) else report
    print("完整复现仍需优先确认：")
    for item in ["index_constituents", "stock_industry", "stock_financials", "stock_daily", "macro_*"]:
        sub = report[report["数据项"].eq(item)]
        if len(sub) and not bool(sub.iloc[0]["是否满足"]):
            print("-", item, "=>", sub.iloc[0]["备注"])
    print("\n近似复现可先用：")
    print("- stock_daily: close_adj/amount/turnover 可先跑价格与流动性；total_mv/float_mv 缺口会影响 Barra 权重和规模中性化。")
    print("- stock_industry: 若只有当前行业，标记 approx_current_industry，仅用于近似行业控制。")
    print("- stock_financials: 无 ann_date 时用 ann_date_approx；ETOP 可用 net_profit_ttm / total_mv，EGRO 可用净利润 TTM 增速近似。")
    print("- etf_daily: nav 缺失时用 close_adj，仍无 close_adj 时用 close 近似执行价。")
    print("\n可安全覆盖正式文件的条件：")
    print("- 候选文件 pandas 可读，字段完整，覆盖至少 2020-2026，关键字段非空率达标，且口径经人工确认。")
    print("- 本 notebook 默认不会覆盖；确认后把 OVERWRITE_CANONICAL=True 并运行覆盖单元。")

print_final_conclusion(final_gap_report)


In [ ]:
def overwrite_if_confirmed(candidate, canonical, required_columns):
    if not OVERWRITE_CANONICAL:
        print("OVERWRITE_CANONICAL=False，跳过覆盖", canonical)
        return False
    candidate = Path(candidate)
    canonical = Path(canonical)
    if not candidate.exists():
        print("candidate missing", candidate)
        return False
    df, err = safe_read_parquet_pandas(candidate)
    if err is not None:
        print("candidate unreadable", candidate, err)
        return False
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        print("candidate missing columns", missing)
        return False
    backup_dir = RAW / "_backup_before_overwrite"
    backup_dir.mkdir(parents=True, exist_ok=True)
    if canonical.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        shutil.copy2(canonical, backup_dir / f"{canonical.stem}.{stamp}{canonical.suffix}")
    shutil.copy2(candidate, canonical)
    print("overwritten", canonical)
    return True

if OVERWRITE_CANONICAL:
    overwrite_if_confirmed(RAW / "stock_daily_candidate.parquet", RAW / "stock_daily.parquet", STOCK_DAILY_COLUMNS)
    overwrite_if_confirmed(RAW / "stock_industry_candidate.parquet", RAW / "stock_industry.parquet", STOCK_INDUSTRY_COLUMNS)
    overwrite_if_confirmed(RAW / "index_constituents_candidate.parquet", RAW / "index_constituents.parquet", INDEX_CONSTITUENTS_COLUMNS)
    overwrite_if_confirmed(RAW / "stock_financials_candidate.parquet", RAW / "stock_financials.parquet", FINANCIAL_COLUMNS)
    overwrite_if_confirmed(RAW / "etf_daily_candidate.parquet", RAW / "etf_daily.parquet", ETF_COLUMNS)
else:
    print("覆盖正式文件开关关闭。")
